# Unidad 3 - RAG Agéntico: Alfred, el Anfitrión Perfecto

En esta unidad construiremos un agente completo que ayuda a **Alfred** a organizar la gala más extravagante del siglo, usando **Retrieval-Augmented Generation (RAG) Agéntico**.

**Temas cubiertos:**
1. Conceptos de RAG y RAG Agéntico
2. Herramientas auxiliares: búsqueda web, clima y estadísticas del Hub
3. Herramienta RAG personalizada para información de invitados (BM25)
4. Ensamblaje del agente Alfred con todas las herramientas
5. Ejemplos de uso y conversación con memoria

**Framework principal:** smolagents (con referencias a LlamaIndex y LangGraph)

---
📖 Referencia: [HuggingFace Agents Course - Unit 3](https://huggingface.co/learn/agents-course/es/unit3/agentic-rag/introduction)

## 1. Conceptos: RAG y RAG Agéntico

Los LLMs se entrenan sobre grandes volúmenes de datos para aprender conocimiento general. Sin embargo, **su conocimiento puede no estar actualizado** o puede no incluir información privada/específica de tu dominio.

**RAG (Retrieval-Augmented Generation)** resuelve esto recuperando información relevante de tus datos y pasándosela al LLM como contexto.

![RAG](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/llama-index/rag.png)

**RAG Agéntico** va un paso más allá: en lugar de recuperar automáticamente contexto de documentos, el **agente decide cuándo y qué herramienta usar** para responder la pregunta.

![Agentic RAG](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/llama-index/agentic-rag.png)

### Los requisitos de la Gala

Alfred, nuestro agente, necesita:
- **Conocimiento de deportes, cultura y ciencia** (búsqueda web)
- **Información sobre los invitados** (RAG sobre dataset de invitados)
- **Estado del clima** (herramienta meteorológica para los fuegos artificiales)
- **Información sobre creadores de IA** (estadísticas del Hugging Face Hub)

## 2. Instalación y Configuración

In [1]:
%pip install smolagents -U
%pip install ddgs
%pip install langchain-community langchain-text-splitters rank_bm25
%pip install huggingface_hub datasets
%pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from huggingface_hub import login

# Inicia sesión en Hugging Face Hub para acceder a la Serverless Inference API.
# Si tienes el token como variable de entorno (HF_TOKEN) ya estás autenticado.
try:
    login()
except ImportError:
    import os
    login(token=os.environ.get("HF_TOKEN"))

## 3. Herramientas Auxiliares para Alfred

Vamos a crear tres herramientas que Alfred usará durante la gala:
1. **Búsqueda web** (DuckDuckGo) — para acceder a las últimas noticias
2. **Información meteorológica** — para decidir cuándo lanzar los fuegos artificiales
3. **Estadísticas del Hub** — para impresionar a los creadores de IA

### 3.1 Herramienta de Búsqueda Web

In [8]:
from smolagents import DuckDuckGoSearchTool

# Inicializar la herramienta de búsqueda DuckDuckGo
search_tool = DuckDuckGoSearchTool()

# Ejemplo de uso
results = search_tool("¿Quién es el actual Presidente de Francia?")
print(results)

## Search Results

[Presidente de Francia - Wikipedia, la enciclopedia libre](https://es.wikipedia.org/wiki/Presidente_de_Francia)
El presidente de la República Francesa (en francés, Président de la République française), más conocido como presidente de Francia, es el jefe de Estado de Francia, gran maestre de la Legión de Honor y copríncipe de Andorra.

[Anexo:Presidentes de Francia - EcuRed](https://www.ecured.cu/Anexo:Presidentes_de_Francia)
Presidente de la República Francesa (en francés: Président de la République française). Es el jefe de Estado de Francia y además co-príncipe de Andorra, de conjunto con el obispo de la Seo de Urgel. El actual presidente es el socialdemócrata François Hollande.

[Francia: Macron anuncia masivo aumento del gasto en Defensa](https://www.dw.com/es/francia-macron-anuncia-masivo-aumento-del-gasto-en-defensa/a-73263662)
El presidente de Francia, Emmanuel Macron.El presidente de Francia, Emmanuel Macron, informó este domingo (13.07.2025) de un significa

### 3.2 Herramienta de Información Meteorológica

Creamos una herramienta personalizada que simula una API meteorológica. En un caso real podrías usar la API de OpenWeatherMap.

In [9]:
from smolagents import Tool
import random

class WeatherInfoTool(Tool):
    name = "weather_info"
    description = "Obtiene información meteorológica para una ubicación dada. Útil para decidir si el clima es adecuado para actividades al aire libre."
    inputs = {
        "location": {
            "type": "string",
            "description": "La ubicación para la que obtener información meteorológica."
        }
    }
    output_type = "string"

    def forward(self, location: str):
        weather_conditions = [
            {"condition": "Lluvioso", "temp_c": 15},
            {"condition": "Despejado", "temp_c": 25},
            {"condition": "Ventoso", "temp_c": 20},
            {"condition": "Nublado", "temp_c": 18},
        ]
        data = random.choice(weather_conditions)
        return f"Clima en {location}: {data['condition']}, {data['temp_c']}°C"

# Inicializar la herramienta
weather_info_tool = WeatherInfoTool()

# Ejemplo de uso
print(weather_info_tool("París"))
print(weather_info_tool("Madrid"))

Clima en París: Lluvioso, 15°C
Clima en Madrid: Despejado, 25°C


### 3.3 Herramienta de Estadísticas del Hugging Face Hub

Esta herramienta permite a Alfred impresionar a los creadores de IA discutiendo sus modelos más populares.

In [ ]:
from smolagents import Tool
from huggingface_hub import list_models

class HubStatsTool(Tool):
    name = "hub_stats"
    description = "Obtiene el modelo más descargado de un autor específico en Hugging Face Hub."
    inputs = {
        "author": {
            "type": "string",
            "description": "El nombre de usuario del autor/organización del modelo para encontrar modelos."
        }
    }
    output_type = "string"

    def forward(self, author: str):
        try:
            # Listar modelos del autor especificado, ordenados por descargas
            models = list(list_models(author=author, sort="downloads", direction=-1, limit=1))
            
            if models:
                model = models[0]
                return f"El modelo más descargado de {author} es {model.id} con {model.downloads:,} descargas."
            else:
                return f"No se encontraron modelos para el autor {author}."
        except Exception as e:
            return f"Error al obtener modelos para {author}: {str(e)}"

# Inicializar la herramienta
hub_stats_tool = HubStatsTool()

# Ejemplo de uso
print(hub_stats_tool("facebook")) # Ejemplo: Obtener el modelo más descargado de Facebook

Error al obtener modelos para facebook: HfApi.list_models() got an unexpected keyword argument 'direction'


### 3.4 Prueba rápida: Alfred con herramientas básicas

Antes de agregar el retriever RAG, probemos Alfred solo con las herramientas auxiliares.

In [12]:
from smolagents import CodeAgent, InferenceClientModel

# Inicializar el modelo
model = InferenceClientModel()

# Alfred con herramientas básicas
alfred_basico = CodeAgent(
    tools=[search_tool, weather_info_tool, hub_stats_tool],
    model=model
)

# Consulta de prueba
response = alfred_basico.run("¿Qué es Facebook y cuál es su modelo más popular en HuggingFace?")
print("🎩 Respuesta de Alfred:")
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ¿Qué es Facebook y cuál es su modelo más popular en HuggingFace?                                                │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: 
Root=1-6a123921-7685e89800a269e34b8d475b;6eb0b894-5d65-436d-97f5-d6596881aca6)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 1: Duration 0.24 seconds]

AgentGenerationError: Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a123921-7685e89800a269e34b8d475b;6eb0b894-5d65-436d-97f5-d6596881aca6)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

## 4. RAG para Información de Invitados

La funcionalidad más importante de Alfred: poder responder preguntas sobre los invitados de la gala en tiempo real.

### ¿Por qué RAG para la Gala?

Un LLM tradicional no tiene información sobre:
1. La lista de invitados específica de tu evento
2. Sus relaciones contigo, intereses y datos de contacto
3. Información que puede actualizarse frecuentemente

**RAG** resuelve esto recuperando la información exacta bajo demanda.

### 4.1 Cargar y Preparar el Dataset de Invitados

In [13]:
import datasets
from langchain_core.documents import Document

# Cargar el dataset de invitados desde Hugging Face
guest_dataset = datasets.load_dataset("agents-course/unit3-invitees", split="train")

print(f"Total de invitados: {len(guest_dataset)}")
print("\nPrimeros 2 invitados:")
for i in range(min(2, len(guest_dataset))):
    guest = guest_dataset[i]
    print(f"  Nombre: {guest['name']}")
    print(f"  Relación: {guest['relation']}")
    print(f"  Email: {guest['email']}")
    print()

Total de invitados: 3

Primeros 2 invitados:
  Nombre: Ada Lovelace
  Relación: best friend
  Email: ada.lovelace@example.com

  Nombre: Dr. Nikola Tesla
  Relación: old friend from university days
  Email: nikola.tesla@gmail.com



In [14]:
# Convertir cada entrada del dataset en un Document de LangChain
# Esto facilita la indexación y recuperación de información
docs = [
    Document(
        page_content="\n".join([
            f"Name: {guest['name']}",
            f"Relation: {guest['relation']}",
            f"Description: {guest['description']}",
            f"Email: {guest['email']}"
        ]),
        metadata={"name": guest["name"]}
    )
    for guest in guest_dataset
]

print(f"Documentos creados: {len(docs)}")
print("\nEjemplo de documento:")
print(docs[0].page_content)

Documentos creados: 3

Ejemplo de documento:
Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com


### 4.2 Crear el Retriever con BM25

Usamos **BM25** (Best Matching 25), un algoritmo clásico de recuperación de texto que no requiere embeddings. Es rápido, efectivo y no necesita GPU.

> **Nota:** Para búsqueda semántica más avanzada, podrías usar retrievers basados en embeddings como los de [sentence-transformers](https://www.sbert.net/) o una base de datos vectorial (ChromaDB, FAISS, etc.).

In [15]:
from smolagents import Tool
from langchain_community.retrievers import BM25Retriever

class GuestInfoRetrieverTool(Tool):
    name = "guest_info_retriever"
    description = (
        "Recupera información detallada sobre los invitados de la gala según su nombre o relación con el anfitrión. "
        "Incluye nombre, relación, descripción y email del invitado."
    )
    inputs = {
        "query": {
            "type": "string",
            "description": "El nombre o descripción del invitado sobre quien se quiere información."
        }
    }
    output_type = "string"

    def __init__(self, docs):
        self.is_initialized = False
        self.retriever = BM25Retriever.from_documents(docs)

    def forward(self, query: str):
        results = self.retriever.invoke(query)
        if results:
            return "\n\n".join([doc.page_content for doc in results[:3]])
        else:
            return "No se encontró información sobre ese invitado."

# Inicializar la herramienta con los documentos del dataset
guest_info_tool = GuestInfoRetrieverTool(docs)

# Prueba directa del retriever
print("🔍 Búsqueda: 'Lady Ada Lovelace'")
print(guest_info_tool("Lady Ada Lovelace"))

🔍 Búsqueda: 'Lady Ada Lovelace'
Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com

Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@example.com

Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com


In [16]:
# Otra prueba del retriever
print("🔍 Búsqueda: 'Tesla wireless energy'")
print(guest_info_tool("Tesla wireless energy"))

🔍 Búsqueda: 'Tesla wireless energy'
Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com

Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@example.com

Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com


### 4.3 Alfred con el RAG de Invitados

Probamos Alfred con la herramienta RAG de invitados antes de agregar todas las herramientas.

In [17]:
from smolagents import CodeAgent, InferenceClientModel

model = InferenceClientModel()

# Alfred solo con el retriever de invitados
alfred_rag = CodeAgent(tools=[guest_info_tool], model=model)

response = alfred_rag.run("Cuéntame sobre nuestra invitada llamada 'Lady Ada Lovelace'.")
print("🎩 Respuesta de Alfred:")
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Cuéntame sobre nuestra invitada llamada 'Lady Ada Lovelace'.                                                    │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: 
Root=1-6a123b29-125f7fe52fed49423263925a;9eae0485-2fbe-4d7d-b2b2-6becb653e2d4)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 1: Duration 0.19 seconds]

AgentGenerationError: Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a123b29-125f7fe52fed49423263925a;9eae0485-2fbe-4d7d-b2b2-6becb653e2d4)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

## 5. Alfred Completo: Ensamblaje de Todas las Herramientas

Ahora combinamos todas las herramientas en un único agente poderoso:
- `guest_info_tool` — información de invitados (RAG/BM25)
- `weather_info_tool` — clima para los fuegos artificiales
- `hub_stats_tool` — estadísticas del HuggingFace Hub
- `search_tool` — búsqueda web general

In [18]:
from smolagents import CodeAgent, InferenceClientModel

# Inicializar el modelo
model = InferenceClientModel()

# Alfred completo con todas las herramientas
alfred = CodeAgent(
    tools=[guest_info_tool, weather_info_tool, hub_stats_tool, search_tool],
    model=model,
    add_base_tools=True,   # incluye herramientas base adicionales
    planning_interval=3    # Alfred planifica cada 3 pasos
)

print("✅ Alfred está listo para la gala con", len(alfred.tools), "herramientas disponibles.")
print("Herramientas:", list(alfred.tools.keys()))

✅ Alfred está listo para la gala con 6 herramientas disponibles.
Herramientas: ['guest_info_retriever', 'weather_info', 'hub_stats', 'web_search', 'visit_webpage', 'final_answer']


## 6. Ejemplos de Uso: Alfred en Acción

### Ejemplo 1: Información sobre un Invitado

In [19]:
response = alfred.run("Cuéntame sobre nuestra invitada 'Lady Ada Lovelace'.")

print("🎩 Respuesta de Alfred:")
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Cuéntame sobre nuestra invitada 'Lady Ada Lovelace'.                                                            │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a123b69-7eeaf9dc68e5d06b4f641dc0;02513722-bbd4-45b4-9947-897e5d02d3c1)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

### Ejemplo 2: Verificar el Clima para los Fuegos Artificiales

In [20]:
response = alfred.run(
    "¿Cómo está el clima en París esta noche? ¿Será adecuado para nuestro espectáculo de fuegos artificiales?"
)

print("🎩 Respuesta de Alfred:")
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ¿Cómo está el clima en París esta noche? ¿Será adecuado para nuestro espectáculo de fuegos artificiales?        │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a123b6f-458ff0a275aa3dc56db10157;c5798846-59c5-4cd6-ab09-c3f918f5fa31)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

### Ejemplo 3: Impresionar a Creadores de IA

In [21]:
response = alfred.run(
    "Uno de nuestros invitados es de Qwen. ¿Qué puedes decirme sobre su modelo más popular?"
)

print("🎩 Respuesta de Alfred:")
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Uno de nuestros invitados es de Qwen. ¿Qué puedes decirme sobre su modelo más popular?                          │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a123b74-13b8b25a00c7041704106177;a1603977-e376-4d18-8527-2cb5ee324be8)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

### Ejemplo 4: Combinando Múltiples Herramientas

Alfred puede combinar el retriever de invitados con búsqueda web para preparar una conversación.

In [22]:
response = alfred.run(
    "Necesito hablar con 'Dr. Nikola Tesla' sobre los avances recientes en energía inalámbrica. "
    "¿Puedes ayudarme a prepararme para esta conversación?"
)

print("🎩 Respuesta de Alfred:")
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Necesito hablar con 'Dr. Nikola Tesla' sobre los avances recientes en energía inalámbrica. ¿Puedes ayudarme a   │
│ prepararme para esta conversación?                                                                              │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a123b7d-12e269ba0ba9930e2d3421d7;7e799f35-3322-4ade-aa76-9f0856f63032)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

## 7. Memoria Conversacional

Alfred puede mantener el contexto de conversaciones anteriores usando `reset=False`. Esto le permite recordar interacciones previas dentro de la misma sesión.

In [23]:
# Primera interacción
response1 = alfred.run("Cuéntame sobre Lady Ada Lovelace.")
print("🎩 Primera respuesta de Alfred:")
print(response1)
print()

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Cuéntame sobre Lady Ada Lovelace.                                                                               │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a123c4d-26e8613828a3be0d25d96f14;5fe5bed1-9d3c-43fe-b879-4cd285726f67)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

In [24]:
# Segunda interacción — Alfred recuerda la anterior con reset=False
response2 = alfred.run(
    "¿En qué proyectos está trabajando actualmente?",
    reset=False  # mantiene el historial de la conversación anterior
)
print("🎩 Segunda respuesta de Alfred (con memoria):")
print(response2)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ¿En qué proyectos está trabajando actualmente?                                                                  │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a123c50-639055246661d8f57b27b5f2;f9a25acd-a7c1-4eb4-baa6-d3b415c80a85)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

## 9. Comparativa: smolagents vs LlamaIndex vs LangGraph

Los tres frameworks cubiertos en el curso permiten implementar RAG Agéntico. Aquí un resumen de las diferencias clave:

| Característica | smolagents | LlamaIndex | LangGraph |
|---|---|---|---|
| **Estilo de agente** | CodeAgent (escribe código Python) | AgentWorkflow | StateGraph (grafo explícito) |
| **Retriever** | BM25Retriever (langchain) | BM25Retriever (llama_index) | BM25Retriever (langchain) |
| **Memoria** | `reset=False` | Context object | Historial de mensajes |
| **Planificación** | `planning_interval` | Automática | Condicional en grafo |
| **Complejidad** | Baja | Media | Alta |

### Versión LlamaIndex (referencia)

```python
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.schema import Document
from llama_index.core.tools import FunctionTool

# Crear documentos
docs_li = [
    Document(
        text=f"Name: {guest['name']}\nRelation: {guest['relation']}\nDescription: {guest['description']}\nEmail: {guest['email']}",
        metadata={"name": guest['name']}
    )
    for guest in guest_dataset
]

bm25_retriever = BM25Retriever.from_defaults(nodes=docs_li)

def get_guest_info(query: str) -> str:
    """Recupera información detallada sobre invitados de la gala según su nombre o relación."""
    results = bm25_retriever.retrieve(query)
    return "\n\n".join([doc.text for doc in results[:3]]) if results else "No encontrado."

guest_tool_li = FunctionTool.from_defaults(get_guest_info)

llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")
alfred_li = AgentWorkflow.from_tools_or_functions([guest_tool_li], llm=llm)

# response = await alfred_li.run("Cuéntame sobre Lady Ada Lovelace.")
```

### Versión LangGraph (referencia)

```python
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.prebuilt import ToolNode
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_community.retrievers import BM25Retriever
from langchain_core.tools import Tool

bm25_retriever = BM25Retriever.from_documents(docs)

def extract_text(query: str) -> str:
    """Recupera información detallada sobre invitados de la gala según su nombre o relación."""
    results = bm25_retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in results[:3]]) if results else "No encontrado."

guest_tool_lg = Tool(name="guest_info_retriever", func=extract_text,
                     description="Recupera información sobre invitados de la gala.")

# ... (grafo StateGraph igual que en la Unidad 2.3)
```